In [1]:
import numpy as np
import os, random
from smplx import SMPLX
import torch
import matplotlib.pyplot as plt
from pathlib import Path
import ipywidgets as widgets
import pyrender
import trimesh
import imageio.v2 as imageio

from glob import glob
from IPython.display import display
from PIL import Image, ImageDraw
from multiprocessing import Pool
from tqdm import tqdm
import glob

In [2]:
from plot_gif_322 import process_smplx_322_data, get_smplx_layer, look_at
from trimesh.transformations import rotation_matrix, translation_matrix

In [3]:
def list_files_in_dir(folder, suffix=".npy"):
    return [
        entry.path
        for entry in os.scandir(folder)
        if entry.is_file() and entry.name.endswith(suffix)
    ]


def random_file_in_dir(all_paths=None, folder=None, suffix=".npy"):
    if all_paths is not None:
        return random.choice(all_paths)

    chosen = None
    count = 0
    with os.scandir(folder) as it:
        for entry in it:
            if entry.is_file() and entry.name.endswith(suffix):
                count += 1
                if random.randrange(count) == 0:
                    chosen = entry.path
    return chosen
dataset_name = "xmopipe_2310"
f = random_file_in_dir(f"./motion_data/smplx_322/{dataset_name}", ".npy")
print(f)

/


In [4]:
def _add_disks(
    scene,
    transl,
    diameter=0.025,
    height=0.025,
    color=(0.05, 0.05, 0.05),
    sections=24,
    min_y=0,
    interpolate=True,
    n_interp=5,
):
    positions = (
        transl.detach().float().cpu().numpy()
        if isinstance(transl, torch.Tensor)
        else np.asarray(transl, dtype=np.float32)
    )
    if positions.size == 0:
        return

    if interpolate and len(positions) > 1:
        interp_positions = []
        for i in range(len(positions) - 1):
            p0, p1 = positions[i], positions[i + 1]
            ctrl = (p0 + p1) / 2 + [0, 0.02, 0]
            interp_positions.append(p0)
            for t in np.linspace(0.2, 0.8, n_interp):
                pt = (1 - t) ** 2 * p0 + 2 * (1 - t) * t * ctrl + t**2 * p1
                interp_positions.append(pt)
        interp_positions.append(positions[-1])
        positions = np.array(interp_positions)

    positions[:, 1] = float(min_y + 0.05)
    radius = float(diameter) / 2.0
    base = trimesh.creation.cylinder(radius=radius, height=height, sections=sections)
    R = trimesh.transformations.rotation_matrix(np.pi / 2.0, [1, 0, 0])
    base.apply_transform(R)

    V0, F0 = base.vertices.astype(np.float32), base.faces.astype(np.int32)
    M, K = len(V0), len(F0)
    N = positions.shape[0]

    verts = (V0[None, :, :] + positions[:, None, :]).reshape(N * M, 3)
    faces = (F0[None, :, :] + (np.arange(N) * M)[:, None, None]).reshape(N * K, 3)
    vcols = np.tile([*color, 1.0], (N * M, 1))

    disks = trimesh.Trimesh(
        vertices=verts, faces=faces, vertex_colors=vcols, process=False
    )
    scene.add(pyrender.Mesh.from_trimesh(disks, smooth=False))

In [5]:
def process_file(example_path, output_dir="gifs_visu"):
    os.makedirs(output_dir, exist_ok=True)

    data = torch.tensor(np.load(example_path)).unsqueeze(0)
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

    smplx_layer, smplx_model = get_smplx_layer(device)
    vert, joints, pose, faces = process_smplx_322_data(
        data, smplx_layer, smplx_model, device=device
    )

    V = vert.squeeze(0).detach().cpu().numpy()
    min_y = V[:, :, 1].min() + pose[0, 310].item()
    F = faces

    scene = pyrender.Scene(ambient_light=[0.3, 0.3, 0.3])

    axis_len = 0.2
    axis_rad = 0.005
    T = translation_matrix([0, 0, axis_len / 2.0])
    Rz = rotation_matrix(np.deg2rad(-90), [0, 0, 1])
    Rx = rotation_matrix(np.deg2rad(90), [0, 1, 0])
    Ry = rotation_matrix(np.deg2rad(-90), [1, 0, 0])

    cyl = trimesh.creation.cylinder(radius=axis_rad, height=axis_len, sections=3)
    blue = pyrender.MetallicRoughnessMaterial(baseColorFactor=[0.0, 0.0, 1.0, 1.0])
    red = pyrender.MetallicRoughnessMaterial(baseColorFactor=[1.0, 0.0, 0.0, 1.0])
    green = pyrender.MetallicRoughnessMaterial(baseColorFactor=[0.0, 1.0, 0.0, 1.0])

    cube_size = (5, 0.1, 5)
    cubes = []
    for i in range(10):
        for j in range(10):
            x = -25 + i * cube_size[0]
            z = -25 + j * cube_size[2]
            transform = np.array(
                [[1, 0, 0, x], [0, 1, 0, min_y], [0, 0, 1, z], [0, 0, 0, 1]]
            )
            cube = trimesh.creation.box(extents=cube_size, transform=transform)
            color = (0.8, 0.9, 0.8) if (i + j) % 2 == 0 else (0.9, 1.0, 0.9)
            cube.visual.vertex_colors = [list(color) + [1.0]] * len(cube.vertices)
            cubes.append(cube)

    merged_ground = trimesh.util.concatenate(cubes)
    scene.add(pyrender.Mesh.from_trimesh(merged_ground, smooth=False))
    transl = pose[:, 309:312]
    _add_disks(
        scene, transl, diameter=0.05, height=0.02, min_y=min_y, color=(0.2, 0.2, 0.2)
    )

    light = pyrender.DirectionalLight(color=[1.0, 1.0, 1.0], intensity=3.0)
    renderer = pyrender.OffscreenRenderer(viewport_width=640, viewport_height=640)
    cam = pyrender.PerspectiveCamera(yfov=np.pi / 4.0)
    eye = (3.5, 2.0 - min_y, 7.5)
    target = (-1.0, 0.25, 0.0)
    cam_pos = look_at(eye, target, up=(0, 1, 0))
    sun_pos = np.array([0.0, 10.0, 0.0])
    light_pose = look_at(sun_pos, target, up=(0, 1, 0))
    scene.add(light, pose=light_pose)

    other_light = pyrender.DirectionalLight(color=[1.0, 1.0, 1.0], intensity=1.0)
    scene.add(other_light, pose=cam_pos)
    scene.add(cam, pose=cam_pos)

    stem = Path(example_path).stem
    dataset_name = "xmopipe_2310"
    label_file = Path(f"{dataset_name}/texts/semantic_labels/{dataset_name}/{stem}.txt")

    if label_file.exists():
        with open(label_file, "r", encoding="utf-8") as f:
            label_text = f.readline().strip()
    else:
        label_text = stem
    frames = []

    for t in range(V.shape[0]):
        v = V[t]
        tm = trimesh.Trimesh(vertices=v, faces=F, process=False)
        color = np.array([120 / 255, 200 / 255, 255 / 255, 1.0], dtype=np.float32)
        tm.visual.vertex_colors = np.tile(color, (len(tm.vertices), 1))

        pm = pyrender.Mesh.from_trimesh(tm, smooth=True)
        node = scene.add(pm)
        axis_x = scene.add(
            pyrender.Mesh.from_trimesh(cyl.copy(), material=red, smooth=True),
            pose=Rx @ T,
        )
        axis_y = scene.add(
            pyrender.Mesh.from_trimesh(cyl.copy(), material=green, smooth=True),
            pose=Ry @ T,
        )
        axis_z = scene.add(
            pyrender.Mesh.from_trimesh(cyl.copy(), material=blue, smooth=True),
            pose=Rz @ T,
        )

        color, _ = renderer.render(
            scene,
            flags=pyrender.RenderFlags.ALL_SOLID
            | pyrender.RenderFlags.SKIP_CULL_FACES
            | pyrender.RenderFlags.SHADOWS_DIRECTIONAL,
        )

        scene.remove_node(axis_x)
        scene.remove_node(axis_y)
        scene.remove_node(axis_z)
        scene.remove_node(node)
        frame_img = Image.fromarray(color)
        ImageDraw.Draw(frame_img).text((10, 10), label_text, fill="black")
        frames.append(np.array(frame_img))

    with imageio.get_writer(f"{output_dir}/{stem}.gif", mode="I", fps=30, loop=0) as writer:
        for frame in frames:
            writer.append_data(frame)

    renderer.delete()
    print(f"GIF saved >>> {output_dir}/{stem}.gif (30 FPS)")

In [ ]:
all_paths = [
    # "./000000.npy",
    "./motion_data/smplx_322/test_no_tolerance/4418_13_0.npy",
    "./motion_data/smplx_322/test_no_tolerance/4422_26_0.npy",
    "./motion_data/smplx_322/test_no_tolerance/4424_4_2.npy",
    "./motion_data/smplx_322/test_no_tolerance/4497_4_0.npy",
    "./motion_data/smplx_322/test_no_tolerance/4494_1_0.npy",
]
random_paths = all_paths
with Pool(processes=4) as p:
    list(
        tqdm(
            p.starmap(process_file, [(path, "gifs_visu") for path in random_paths]),
            total=len(random_paths),
        )
    )

FileNotFoundError: [Errno 2] No such file or directory: './motion_data/smplx_322/test_no_tolerance/4497_4_0.npy'

In [6]:
all_paths = list_files_in_dir(
    "./xmopipe_2310/motion_data/smplx_322/xmopipe_2310", ".npy"
)
random_paths = [random.choice(all_paths) for _ in range(50)]

with Pool(processes=4) as p:
    list(
        tqdm(
            p.starmap(process_file, [(path, "gifs_visu") for path in random_paths]),
            total=len(random_paths),
        )
    )

GIF saved >>> gifs_visu/10730_9_0.gif (30 FPS)
GIF saved >>> gifs_visu/12034_5_0.gif (30 FPS)
GIF saved >>> gifs_visu/15273_79_0.gif (30 FPS)
GIF saved >>> gifs_visu/10727_15_0.gif (30 FPS)
GIF saved >>> gifs_visu/12016_31_0.gif (30 FPS)
GIF saved >>> gifs_visu/17754_64_1.gif (30 FPS)
GIF saved >>> gifs_visu/18080_35_1.gif (30 FPS)
GIF saved >>> gifs_visu/12425_57_0.gif (30 FPS)
GIF saved >>> gifs_visu/18123_15_5.gif (30 FPS)
GIF saved >>> gifs_visu/16427_13_1.gif (30 FPS)
GIF saved >>> gifs_visu/13292_7_2.gif (30 FPS)
GIF saved >>> gifs_visu/18879_17_0.gif (30 FPS)
GIF saved >>> gifs_visu/18978_15_0.gif (30 FPS)
GIF saved >>> gifs_visu/10716_25_0.gif (30 FPS)
GIF saved >>> gifs_visu/15829_6_0.gif (30 FPS)
GIF saved >>> gifs_visu/17443_34_3.gif (30 FPS)
GIF saved >>> gifs_visu/15445_34_0.gif (30 FPS)
GIF saved >>> gifs_visu/12930_122_1.gif (30 FPS)
GIF saved >>> gifs_visu/15329_26_1.gif (30 FPS)
GIF saved >>> gifs_visu/17825_68_0.gif (30 FPS)
GIF saved >>> gifs_visu/11070_252_0.gif (30

UnicodeEncodeError: 'latin-1' codec can't encode character '\u2019' in position 32: ordinal not in range(256)